In [3]:
import os
import json

dataset_path = "data/manifestations_raw"
all_ttps = []

for step_dir in os.listdir(os.path.join(dataset_path, "steps")):
    step_path = os.path.join(dataset_path, "steps", step_dir)
    if os.path.exists(os.path.join(step_path, "attacker/logs/attackmate.json")):
        ttps = []
        with open(os.path.join(step_path, "attacker/logs/attackmate.json")) as f:
            for l in f:
                attackmate_data = json.loads(l)
                param = attackmate_data.get("parameters", {})
                metadata = param.get("metadata", {}) if param else {}
                techniques = metadata.get("techniques", []) if metadata else []

                possible_ttps = techniques.strip().split(",") if isinstance(techniques, str) else []
                if len(possible_ttps) > 0:
                    ttps.append(possible_ttps)
        all_ttps.append(ttps)

inconsistencies = []
for e in all_ttps:
    if len(e) > 1:
        # Comprobar que los elementos de la lista son iguales
        if not all(x == e[0] for x in e):
            inconsistencies.append(e)
if len(inconsistencies) > 0:
    print("Se han encontrado inconsistencias en los ttps:")
    for inc in inconsistencies:
        print(inc)

else:
    print("No se han encontrado inconsistencias en los ttps.")

# Buscar todos los archivos dentro de un directorio que su ruta acabe en alerts.json
paths = []

for root, dirs, files in os.walk(os.path.join(dataset_path, "steps")):
    for file in files:
        if file == "alerts.json":
            alert_file_path = os.path.join(root, file)
            paths.append(alert_file_path)

different_paths = []
for p in paths:
    if not p.endswith("wazuh\\logs\\alerts\\alerts.json"):
        different_paths.append(p)

if len(different_paths) > 0:
    print("Se han encontrado rutas diferentes a wazuh\\logs\\alerts\\alerts.json:")
    for dp in different_paths:
        print(dp)

else:
    print("Todas las rutas de alertas acaban en wazuh\\logs\\alerts\\alerts.json.")

# Buscar todos los archivos dentro de un directorio que su ruta acabe en alerts.json
paths = []

for root, dirs, files in os.walk(os.path.join(dataset_path, "steps")):
    for file in files:
        if file == "attackmate.json":
            alert_file_path = os.path.join(root, file)
            paths.append(alert_file_path)

different_paths = []
for p in paths:
    if not p.endswith("attacker\\logs\\attackmate.json"):
        different_paths.append(p)

if len(different_paths) > 0:
    print("Se han encontrado rutas diferentes a attacker\\logs\\attackmate.json:")
    for dp in different_paths:
        print(dp)

else:
    print("Todas las rutas de attackmate acaban en attacker\\logs\\attackmate.json.")

No se han encontrado inconsistencias en los ttps.
Todas las rutas de alertas acaban en wazuh\logs\alerts\alerts.json.
Todas las rutas de attackmate acaban en attacker\logs\attackmate.json.


In [10]:
import csv
import json
import os

dataset_path = "data/manifestations_raw"
alerts_with_truth = []
paths_to_save = []

with open('data/alerts_dataset.csv', 'w') as csvfile:
    writer = csv.writer(csvfile)

    writer.writerow(["alert", "real_ttps", "possible_ttps", "scenario", "step_dir", "alert_path"])  
    
    for step_dir in os.listdir(os.path.join(dataset_path, "steps")):
        step_path = os.path.join(dataset_path, "steps", step_dir)
        scenario = int(step_dir[0])
        if os.path.exists(os.path.join(step_path, "attacker/logs/attackmate.json")):
            paths_to_save.append(os.path.join(step_path, "attacker/logs/attackmate.json"))
            ttps = []
            with open(os.path.join(step_path, "attacker/logs/attackmate.json")) as f:
                for l in f:
                    attackmate_data = json.loads(l)
                    param = attackmate_data.get("parameters", {})
                    metadata = param.get("metadata", {}) if param else {}
                    techniques = metadata.get("techniques", []) if metadata else []

                    possible_ttps = techniques.strip().split(",") if isinstance(techniques, str) else []
                    if len(possible_ttps) > 0:
                        ttps.extend(possible_ttps)
                        continue  # Si encontramos técnicas, no necesitamos buscar más en este paso (se comprobó que coinciden si hay más de una línea que contenga)

            if os.path.exists(os.path.join(step_path, "wazuh/logs/alerts/alerts.json")):
                paths_to_save.append(os.path.join(step_path, "wazuh/logs/alerts/alerts.json"))
                with open(os.path.join(step_path, "wazuh/logs/alerts/alerts.json")) as f:
                    for line in f:
                        alert_data = json.loads(line)
                        real_ttps = alert_data.get('rule', {}).get('mitre', {}).get('id', [])
                        
                        # alerts_with_truth.append({
                        #     "alert": json.loads(line),
                        #     "real_ttps": real_ttps,
                        #     "possibles_ttps": ttps,
                        #     "path": os.path.join(step_path, "wazuh/logs/alerts/alerts.json")
                        # })
                        writer.writerow([json.dumps(alert_data), real_ttps, ttps, scenario, step_dir, os.path.join(step_path, "wazuh/logs/alerts/alerts.json")])
                

In [1]:
import pandas as pd

df = pd.read_csv('data/alerts_dataset.csv')

df.head()

,alert,real_ttps,possible_ttps,scenario,step_dir,alert_path
0,"{""timestamp"": ""2025-09-22T18:37:00.601+0000"", ...",['T1057'],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
1,"{""timestamp"": ""2025-09-22T18:37:06.758+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
2,"{""timestamp"": ""2025-09-22T18:37:06.774+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
3,"{""timestamp"": ""2025-09-22T18:37:10.786+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
4,"{""timestamp"": ""2025-09-22T18:37:12.713+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...


In [17]:
df[df['step_dir'].str.contains('1_autostart_localaccount')]

,alert,real_ttps,possible_ttps,scenario,step_dir,alert_path
0,"{""timestamp"": ""2025-09-22T18:37:00.601+0000"", ...",['T1057'],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
1,"{""timestamp"": ""2025-09-22T18:37:06.758+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
2,"{""timestamp"": ""2025-09-22T18:37:06.774+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
3,"{""timestamp"": ""2025-09-22T18:37:10.786+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
4,"{""timestamp"": ""2025-09-22T18:37:12.713+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
...,...,...,...,...,...,...
13723,"{""timestamp"": ""2025-09-22T18:31:49.767+0000"", ...",[],['T1595.003'],1,1_autostart_localaccount-4,data/manifestations_raw\steps\1_autostart_loca...
13724,"{""timestamp"": ""2025-09-22T18:31:49.767+0000"", ...",[],['T1595.003'],1,1_autostart_localaccount-4,data/manifestations_raw\steps\1_autostart_loca...
13725,"{""timestamp"": ""2025-09-22T18:31:49.768+0000"", ...",[],['T1595.003'],1,1_autostart_localaccount-4,data/manifestations_raw\steps\1_autostart_loca...
13726,"{""timestamp"": ""2025-09-22T18:32:33.307+0000"", ...",[],['T1595.003'],1,1_autostart_localaccount-4,data/manifestations_raw\steps\1_autostart_loca...


In [3]:
df['real_ttps'].value_counts()

real_ttps
[]                             220584
['T1595.002']                   14209
['T1055', 'T1083', 'T1190']      5201
['T1059.007']                    3791
['T1055', 'T1083']                566
['T1059']                         411
['T1078']                         290
['T1055']                         248
['T1548.003']                     196
['T1057']                          91
['T1190']                          57
['T1078', 'T1021']                 47
['T1499']                          35
['T1210']                          35
['T1083']                          13
['T1136']                          12
['T1562.001']                       3
['T1040']                           2
Name: count, dtype: int64

In [18]:
import json
parsed_alerts = []

for alert in df.alert:
    alert_data = json.loads(alert)
    timestamp = alert_data.get("timestamp", "")
    rule = alert_data.get("rule", {})
    level = rule.get("level", "")
    desc = rule.get("description", "")
    rule_id = rule.get("id", "")
    fired_times = rule.get("firedtimes", "")
    groups = rule.get("groups", [])
    agent = alert_data.get("agent", {})
    manager = alert_data.get("manager", {})
    alert_id = alert_data.get("id", "")
    log = alert_data.get("full_log", "")
    log_data = alert_data.get("data", {})

    parsed_alerts.append([timestamp, level, desc, rule_id, fired_times, groups, agent, manager, alert_id, log, log_data])

alerts_df = pd.DataFrame(columns=["timestamp", "level", "description", "rule_id", "fired_times", "groups", "agent", "manager", "alert_id", "full_log", "data"], data=parsed_alerts)

In [19]:
alerts_with_ttp = df[df['real_ttps'] != '[]']
alerts_with_ttp

,alert,real_ttps,possible_ttps,scenario,step_dir,alert_path
0,"{""timestamp"": ""2025-09-22T18:37:00.601+0000"", ...",['T1057'],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
7,"{""timestamp"": ""2025-09-22T18:37:18.892+0000"", ...",['T1057'],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
9,"{""timestamp"": ""2025-09-22T18:37:20.423+0000"", ...",['T1057'],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
10,"{""timestamp"": ""2025-09-22T18:37:20.424+0000"", ...",['T1057'],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
11,"{""timestamp"": ""2025-09-22T18:37:20.458+0000"", ...",['T1057'],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
...,...,...,...,...,...,...
245496,"{""timestamp"": ""2025-12-15T11:05:46.800+0000"", ...",['T1078'],['T1548.003'],4,4-8,data/manifestations_raw\steps\4-8\wazuh/logs/a...
245499,"{""timestamp"": ""2025-12-15T11:06:02.749+0000"", ...",['T1548.003'],"['T1036.005', 'T1205.001']",4,4-9,data/manifestations_raw\steps\4-9\wazuh/logs/a...
245500,"{""timestamp"": ""2025-12-15T11:06:02.763+0000"", ...",['T1078'],"['T1036.005', 'T1205.001']",4,4-9,data/manifestations_raw\steps\4-9\wazuh/logs/a...
245502,"{""timestamp"": ""2025-12-15T11:06:02.801+0000"", ...",['T1548.003'],"['T1036.005', 'T1205.001']",4,4-9,data/manifestations_raw\steps\4-9\wazuh/logs/a...


In [20]:
alerts_with_ttp.groupby('alert_path').count()

,alert,real_ttps,possible_ttps,scenario,step_dir
alert_path,,,,,
data/manifestations_raw\steps\1_autostart_localaccount-13\wazuh/logs/alerts/alerts.json,6,6,6,6,6
data/manifestations_raw\steps\1_autostart_localaccount-16\wazuh/logs/alerts/alerts.json,40,40,40,40,40
data/manifestations_raw\steps\1_autostart_localaccount-21\wazuh/logs/alerts/alerts.json,2,2,2,2,2
data/manifestations_raw\steps\1_autostart_localaccount-23\wazuh/logs/alerts/alerts.json,3,3,3,3,3
data/manifestations_raw\steps\1_autostart_localaccount-24\wazuh/logs/alerts/alerts.json,4,4,4,4,4
...,...,...,...,...,...
data/manifestations_raw\steps\4-17\wazuh/logs/alerts/alerts.json,4,4,4,4,4
data/manifestations_raw\steps\4-3\wazuh/logs/alerts/alerts.json,3,3,3,3,3
data/manifestations_raw\steps\4-6\wazuh/logs/alerts/alerts.json,6,6,6,6,6


In [21]:
# Convertir timestamp a formato datetime
alerts_df['timestamp'] = pd.to_datetime(alerts_df['timestamp'])
alerts_df

,timestamp,level,description,rule_id,fired_times,groups,agent,manager,alert_id,full_log,data
0,2025-09-22 18:37:00.601000+00:00,6,Processes running for all users were queried w...,92604,1,[audit_detections],"{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566220.11498102,type=SYSCALL msg=audit(1758566218.934:4968): a...,"{'audit': {'type': 'SYSCALL', 'id': '4968', 'a..."
1,2025-09-22 18:37:06.758000+00:00,7,Agent event queue is 90% full.,202,4,"[wazuh, agent_flooding]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566226.11500168,wazuh: Agent buffer: '90%'.,{'level': '90%'}
2,2025-09-22 18:37:06.774000+00:00,9,Agent event queue is full. Events may be lost.,203,7,"[wazuh, agent_flooding]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566226.11500400,wazuh: Agent buffer: 'full'.,{'level': 'full'}
3,2025-09-22 18:37:10.786000+00:00,9,Agent event queue is full. Events may be lost.,203,8,"[wazuh, agent_flooding]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566230.11500650,wazuh: Agent buffer: 'full'.,{'level': 'full'}
4,2025-09-22 18:37:12.713000+00:00,9,Agent event queue is full. Events may be lost.,203,9,"[wazuh, agent_flooding]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566232.11500900,wazuh: Agent buffer: 'full'.,{'level': 'full'}
...,...,...,...,...,...,...,...,...,...,...,...
245786,2025-12-10 17:11:24.157000+00:00,6,IDS event.,20101,46,[ids],"{'id': '002', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1765386684.1769863,12/10/2025-17:11:22.506215 [**] [1:2044076:1]...,"{'srcip': '192.168.50.100', 'dstip': '192.168...."
245787,2025-12-10 17:11:24.159000+00:00,6,IDS event.,20101,47,[ids],"{'id': '002', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1765386684.1770239,12/10/2025-17:11:23.502233 [**] [1:2043343:1]...,"{'srcip': '192.168.50.100', 'dstip': '192.168...."
245788,2025-12-10 17:11:24.161000+00:00,6,IDS event.,20101,48,[ids],"{'id': '002', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1765386684.1770609,12/10/2025-17:11:23.502233 [**] [1:2044076:1]...,"{'srcip': '192.168.50.100', 'dstip': '192.168...."
245789,2025-12-10 17:24:48.705000+00:00,3,Suricata: Alert - ET INFO Python BaseHTTP Serv...,86601,57,"[ids, suricata]","{'id': '002', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1765387488.1773877,,{'timestamp': '2025-12-10T17:24:47.268073+0000...


In [22]:
alerts_df.sort_values(by='timestamp', inplace=True)
alerts_df

,timestamp,level,description,rule_id,fired_times,groups,agent,manager,alert_id,full_log,data
108016,2025-09-17 13:36:28.789000+00:00,3,Suricata: Alert - ET SCAN Nikto Web App Scan i...,86601,75,"[ids, suricata]","{'id': '001', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1758116188.1793674,,{'timestamp': '2025-09-17T13:36:28.440018+0000...
108017,2025-09-17 13:36:29.003000+00:00,5,Web server 400 error code.,31101,1,"[web, accesslog, attack]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758116189.1795863,192.42.1.174 - - [17/Sep/2025:13:36:27 +0000] ...,"{'protocol': 'GET', 'srcip': '192.42.1.174', '..."
108018,2025-09-17 13:36:29.004000+00:00,5,Web server 400 error code.,31101,2,"[web, accesslog, attack]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758116189.1796358,192.42.1.174 - - [17/Sep/2025:13:36:27 +0000] ...,"{'protocol': 'GET', 'srcip': '192.42.1.174', '..."
108019,2025-09-17 13:36:29.005000+00:00,3,Suricata: Alert - GPL EXPLOIT ISAPI .idq access,86601,76,"[ids, suricata]","{'id': '001', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1758116189.1796852,,{'timestamp': '2025-09-17T13:36:28.534392+0000...
108020,2025-09-17 13:36:29.007000+00:00,5,Web server 400 error code.,31101,3,"[web, accesslog, attack]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758116189.1799062,192.42.1.174 - - [17/Sep/2025:13:36:27 +0000] ...,"{'protocol': 'GET', 'srcip': '192.42.1.174', '..."
...,...,...,...,...,...,...,...,...,...,...,...
245476,2025-12-15 11:13:49.045000+00:00,6,IDS event.,20101,24,[ids],"{'id': '001', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1765797229.1009240,12/15/2025-11:13:47.963759 [**] [1:2210059:1]...,"{'srcip': '172.17.100.122', 'dstip': '192.168...."
245477,2025-12-15 11:14:01.094000+00:00,3,PAM: Login session closed.,5502,18,"[pam, syslog]","{'id': '001', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1765797241.1009612,Dec 15 11:14:00 inetfw sshd[4074]: pam_unix(ss...,{'dstuser': 'john'}
245478,2025-12-15 11:14:01.192000+00:00,3,PAM: Login session closed.,5502,19,"[pam, syslog]","{'id': '001', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1765797241.1009990,Dec 15 11:14:00 inetfw sshd[4074]: pam_unix(ss...,{'dstuser': 'john'}
245479,2025-12-15 11:14:03.053000+00:00,3,PAM: Login session closed.,5502,20,"[pam, syslog]","{'id': '001', 'name': 'inetfw', 'ip': '192.168...",{'name': 'wazuh'},1765797243.1010359,Dec 15 11:14:01 inetfw sudo: pam_unix(sudo:ses...,{'dstuser': 'root'}


In [23]:
alerts_df.groupby('description').size()

description
Agent event queue is 90% full.                                72
Agent event queue is back to normal load.                     72
Agent event queue is full. Events may be lost.               242
Apache: Attempt to access forbidden directory index.          35
Apache: Attempt to access forbidden file or directory.       155
                                                           ...  
Wazuh agent started.                                           3
Wazuh agent stopped.                                           3
Web server 400 error code.                                184822
XSS (Cross Site Scripting) attempt.                         3791
sshd: authentication success.                                 47
Length: 164, dtype: int64

In [26]:
df.iloc[349]

alert            {"timestamp": "2025-09-22T18:30:38.258+0000", ...
real_ttps                                                       []
possible_ttps                                        ['T1595.002']
scenario                                                         1
step_dir                                1_autostart_localaccount-3
alert_path       data/manifestations_raw\steps\1_autostart_loca...
Name: 349, dtype: object

In [27]:
df.iloc[349].alert

'{"timestamp": "2025-09-22T18:30:38.258+0000", "rule": {"level": 5, "description": "Web server 400 error code.", "id": "31101", "firedtimes": 162, "mail": false, "groups": ["web", "accesslog", "attack"], "pci_dss": ["6.5", "11.4"], "gdpr": ["IV_35.7.d"], "nist_800_53": ["SA.11", "SI.4"], "tsc": ["CC6.6", "CC7.1", "CC8.1", "CC6.1", "CC6.8", "CC7.2", "CC7.3"]}, "agent": {"id": "002", "name": "videoserver", "ip": "172.17.100.121"}, "manager": {"name": "wazuh"}, "id": "1758565838.1814008", "full_log": "192.42.1.174 - - [22/Sep/2025:18:30:36 +0000] \\"GET /borLjZ2z.shtm HTTP/1.1\\" 404 396 \\"-\\" \\"Mozilla/5.00 (Nikto/2.1.5) (Evasions:None) (Test:map_codes)\\"", "decoder": {"name": "web-accesslog"}, "data": {"protocol": "GET", "srcip": "192.42.1.174", "id": "404", "url": "/borLjZ2z.shtm"}, "location": "/var/www/default/log/access.log"}'

In [2]:
mini_df = pd.read_csv('data/mini_dataset.csv')
mini_df.head()

,alert,real_ttps,possible_ttps,scenario,step_dir,alert_path
0,"{""timestamp"": ""2025-09-22T18:37:00.601+0000"", ...",['T1057'],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
1,"{""timestamp"": ""2025-09-22T18:37:06.758+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
2,"{""timestamp"": ""2025-09-22T18:37:06.774+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
3,"{""timestamp"": ""2025-09-22T18:37:10.786+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...
4,"{""timestamp"": ""2025-09-22T18:37:12.713+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...


In [3]:
import json
parsed_alerts = []

for alert in mini_df.alert:
    alert_data = json.loads(alert)
    timestamp = alert_data.get("timestamp", "")
    rule = alert_data.get("rule", {})
    level = rule.get("level", "")
    desc = rule.get("description", "")
    rule_id = rule.get("id", "")
    fired_times = rule.get("firedtimes", "")
    groups = rule.get("groups", [])
    agent = alert_data.get("agent", {})
    manager = alert_data.get("manager", {})
    alert_id = alert_data.get("id", "")
    log = alert_data.get("full_log", "")
    log_data = alert_data.get("data", {})

    parsed_alerts.append([timestamp, level, desc, rule_id, fired_times, groups, agent, manager, alert_id, log, log_data])

# Añadir a mini_df las nuevas columnas
mini_df[["timestamp", "level", "description", "rule_id", "fired_times", "groups", "agent", "manager", "alert_id", "full_log", "data"]] = pd.DataFrame(parsed_alerts, index=mini_df.index)
# Convertir timestamp a formato datetime
mini_df['timestamp'] = pd.to_datetime(mini_df['timestamp'])
mini_df.sort_values(by='timestamp', inplace=True)
mini_df.head()

,alert,real_ttps,possible_ttps,scenario,step_dir,alert_path,timestamp,level,description,rule_id,fired_times,groups,agent,manager,alert_id,full_log,data
0,"{""timestamp"": ""2025-09-22T18:37:00.601+0000"", ...",['T1057'],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...,2025-09-22 18:37:00.601000+00:00,6,Processes running for all users were queried w...,92604,1,[audit_detections],"{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566220.11498102,type=SYSCALL msg=audit(1758566218.934:4968): a...,"{'audit': {'type': 'SYSCALL', 'id': '4968', 'a..."
1,"{""timestamp"": ""2025-09-22T18:37:06.758+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...,2025-09-22 18:37:06.758000+00:00,7,Agent event queue is 90% full.,202,4,"[wazuh, agent_flooding]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566226.11500168,wazuh: Agent buffer: '90%'.,{'level': '90%'}
2,"{""timestamp"": ""2025-09-22T18:37:06.774+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...,2025-09-22 18:37:06.774000+00:00,9,Agent event queue is full. Events may be lost.,203,7,"[wazuh, agent_flooding]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566226.11500400,wazuh: Agent buffer: 'full'.,{'level': 'full'}
3,"{""timestamp"": ""2025-09-22T18:37:10.786+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...,2025-09-22 18:37:10.786000+00:00,9,Agent event queue is full. Events may be lost.,203,8,"[wazuh, agent_flooding]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566230.11500650,wazuh: Agent buffer: 'full'.,{'level': 'full'}
4,"{""timestamp"": ""2025-09-22T18:37:12.713+0000"", ...",[],"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', ...",1,1_autostart_localaccount-13,data/manifestations_raw\steps\1_autostart_loca...,2025-09-22 18:37:12.713000+00:00,9,Agent event queue is full. Events may be lost.,203,9,"[wazuh, agent_flooding]","{'id': '002', 'name': 'videoserver', 'ip': '17...",{'name': 'wazuh'},1758566232.11500900,wazuh: Agent buffer: 'full'.,{'level': 'full'}


In [4]:
mini_df.to_csv('data/mini_dataset_parsed.csv', index=False)

In [8]:
mini_df = pd.read_csv('data/mini_dataset_parsed.csv')

In [9]:
actual_time = mini_df.iloc[7].timestamp

# Buscar alertas anteriores y posteriores a actual_time
window_size = 10
previous_alerts = mini_df[mini_df['timestamp'] < actual_time].tail(window_size//2)[['timestamp', 'level', 'description', 'fired_times', 'full_log']]
subsequent_alerts = mini_df[mini_df['timestamp'] > actual_time].head(window_size//2)[['timestamp', 'level', 'description', 'fired_times', 'full_log']]

context = f'Alert original:\n{mini_df.iloc[7].alert}\n\nAlertas anteriores:\n{previous_alerts.to_dict(orient="records")}\n\nAlertas posteriores:\n{subsequent_alerts.to_dict(orient="records")}'

In [6]:
print(context)

Alert original:
{"timestamp": "2025-09-22T18:37:18.892+0000", "rule": {"level": 6, "description": "Processes running for all users were queried with ps command.", "id": "92604", "mitre": {"id": ["T1057"], "tactic": ["Discovery"], "technique": ["Process Discovery"]}, "firedtimes": 2, "mail": false, "groups": ["audit_detections"]}, "agent": {"id": "002", "name": "videoserver", "ip": "172.17.100.121"}, "manager": {"name": "wazuh"}, "id": "1758566238.11501650", "full_log": "type=SYSCALL msg=audit(1758566228.862:6639): arch=c000003e syscall=59 success=yes exit=0 a0=55de6e592538 a1=55de6e592478 a2=55de6e592490 a3=8 items=3 ppid=6296 pid=6300 auid=4294967295 uid=33 gid=33 euid=33 suid=33 fsuid=33 egid=33 sgid=33 fsgid=33 tty=pts0 ses=4294967295 comm=\"ps\" exe=\"/usr/bin/ps\" subj==unconfined key=\"T1166_Seuid_and_Setgid\"\u001dARCH=x86_64 SYSCALL=execve AUID=\"unset\" UID=\"www-data\" GID=\"www-data\" EUID=\"www-data\" SUID=\"www-data\" FSUID=\"www-data\" EGID=\"www-data\" SGID=\"www-data\" 

In [7]:
alert_data = json.loads(mini_df.iloc[7].alert)

prompt = f'Alerta: {alert_data}'
print(prompt)

Alerta: {'timestamp': '2025-09-22T18:37:18.892+0000', 'rule': {'level': 6, 'description': 'Processes running for all users were queried with ps command.', 'id': '92604', 'mitre': {'id': ['T1057'], 'tactic': ['Discovery'], 'technique': ['Process Discovery']}, 'firedtimes': 2, 'mail': False, 'groups': ['audit_detections']}, 'agent': {'id': '002', 'name': 'videoserver', 'ip': '172.17.100.121'}, 'manager': {'name': 'wazuh'}, 'id': '1758566238.11501650', 'full_log': 'type=SYSCALL msg=audit(1758566228.862:6639): arch=c000003e syscall=59 success=yes exit=0 a0=55de6e592538 a1=55de6e592478 a2=55de6e592490 a3=8 items=3 ppid=6296 pid=6300 auid=4294967295 uid=33 gid=33 euid=33 suid=33 fsuid=33 egid=33 sgid=33 fsgid=33 tty=pts0 ses=4294967295 comm="ps" exe="/usr/bin/ps" subj==unconfined key="T1166_Seuid_and_Setgid"\x1dARCH=x86_64 SYSCALL=execve AUID="unset" UID="www-data" GID="www-data" EUID="www-data" SUID="www-data" FSUID="www-data" EGID="www-data" SGID="www-data" FSGID="www-data" type=EXECVE msg

In [21]:
mini_df.iloc[8].possible_ttps

"['T1087', 'T1083', 'T1201', 'T1069', 'T1057', 'T1518', 'T1082', 'T1614', 'T1016', 'T1049', 'T1033', 'T1007', 'T1615']"